# Day 3 — Single-agent baseline

First, validate a manually constructed report against collected evidence.
Then replace manual report construction with a tool-using agent.

The football dataset contains fictional test data.

In [1]:
import json

from aegis.schemas import (
    ComparisonRow,
    FinalReport,
    ResearchBundle,
    validate_report_against_bundle,
)
from aegis.tools.evidence import lookup_evidence

In [2]:
# Retrieve scoring evidence using our existing deterministic tool.
# Each lookup returns both seasons for the requested player.
lookups = [
    lookup_evidence(
        subject=subject,
        factor="scoring",
        dataset_ref="football_demo_v1",
    )
    for subject in ["Player A", "Player B"]
]

# Both players belong to the same source dataset.
# Deduplicate sources by ID before constructing the bundle.
sources_by_id = {}
evidence_by_id = {}

for lookup in lookups:
    if lookup.source is None or not lookup.evidence:
        raise ValueError(
            f"Expected fixture evidence, received: {lookup.reason}"
        )

    sources_by_id[lookup.source.source_id] = lookup.source

    for item in lookup.evidence:
        evidence_by_id[item.evidence_id] = item

# ResearchBundle checks that the evidence references valid sources.
bundle = ResearchBundle(
    sources=list(sources_by_id.values()),
    evidence=list(evidence_by_id.values()),
)

print(f"Sources: {len(bundle.sources)}")
print(f"Evidence items: {len(bundle.evidence)}")

Sources: 1
Evidence items: 4


In [3]:
# Select the requested season explicitly so we don't mix periods.
selected_evidence = [
    item
    for item in bundle.evidence
    if item.time_period == "2025"
]

comparison_rows = []

for item in selected_evidence:
    # The fixture tool preserves its selected fields as JSON.
    observation = json.loads(item.excerpt)

    comparison_rows.append(
        ComparisonRow(
            subject=item.subject,
            factor="scoring",
            value=observation["goals"],
            unit="goals",
            evidence_ids=[item.evidence_id],
        )
    )

# This is a handwritten report, not an LLM-generated response.
# The numerical rows come from retrieved fixture evidence.
report = FinalReport(
    summary=(
        "In the fictional 2025 Demo League fixture, "
        "Player A recorded 14 goals and Player B recorded 11."
    ),
    mode="compare",
    subjects=["Player A", "Player B"],
    comparison_rows=comparison_rows,
    ranking=[],
    findings=[],
    limitations=[
        "Synthetic test data; these are not real player statistics.",
        "Comparison covers only goals in the 2025 Demo League.",
        "Goal totals do not account for differences in playing time.",
    ],
    citations=[source.source_id for source in bundle.sources],
    status="completed",
)

# Check that every referenced evidence item and source exists.
validated_report = validate_report_against_bundle(report, bundle)

print(validated_report.model_dump_json(indent=2))
print("PASS: handwritten report validates against fixture evidence")

{
  "summary": "In the fictional 2025 Demo League fixture, Player A recorded 14 goals and Player B recorded 11.",
  "mode": "compare",
  "subjects": [
    "Player A",
    "Player B"
  ],
  "comparison_rows": [
    {
      "subject": "Player A",
      "factor": "scoring",
      "value": 14.0,
      "unit": "goals",
      "evidence_ids": [
        "ev_42aa13f70da1973f10be907aa173e5b20984b0b3f94dd6d6671edb4fa10a6519"
      ]
    },
    {
      "subject": "Player B",
      "factor": "scoring",
      "value": 11.0,
      "unit": "goals",
      "evidence_ids": [
        "ev_ca42f0862bb7a926e9972fb79e7e95b69a9621a0d45352a94d1c624fc84ef972"
      ]
    }
  ],
  "ranking": [],
  "findings": [],
  "limitations": [
    "Synthetic test data; these are not real player statistics.",
    "Comparison covers only goals in the 2025 Demo League.",
    "Goal totals do not account for differences in playing time."
  ],
  "citations": [
    "src_football_demo_v1"
  ],
  "status": "completed"
}
PASS: handwri

## Experiment A: model-driven tool use

The model selects tools and constructs the report.
Python validates the report against evidence actually returned by tools.

This experiment uses fictional local data, not web research.

In [4]:
import json
import os
from decimal import Decimal
from pathlib import Path

from dotenv import load_dotenv

# Find the project root whether the notebook starts in the root
# directory or inside notebooks/.
project_root = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "pyproject.toml").is_file()
        and (path / "src" / "aegis").is_dir()
    ),
    None,
)

if project_root is None:
    raise RuntimeError("Could not locate the Aegis project root")

# Export .env values so LangSmith can read them from os.environ.
load_dotenv(project_root / ".env", override=True)

# Settings uses a relative .env path, so use the project root.
os.chdir(project_root)

from aegis.config import get_settings

# Discard settings cached before the environment was loaded.
get_settings.cache_clear()
settings = get_settings()

# Print configuration status, never credentials.
print("Tracing enabled:", os.getenv("LANGSMITH_TRACING"))
print("LangSmith project:", os.getenv("LANGSMITH_PROJECT"))
print("LangSmith key present:", bool(os.getenv("LANGSMITH_API_KEY")))

role = settings.role_models["researcher"]
print("Researcher provider:", role.provider)
print("Researcher model:", role.model)

Tracing enabled: true
LangSmith project: aegis-dev
LangSmith key present: True
Researcher provider: azure
Researcher model: gpt-5-mini


In [5]:
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_core.messages import AIMessage, ToolMessage

from aegis.clarification import check_clarity
from aegis.llm.callbacks import new_run_budget
from aegis.llm.gateway import get_model
from aegis.schemas import (
    FinalReport,
    ResearchBundle,
    ResearchRequest,
    ResearchScope,
    validate_report_against_bundle,
)
from aegis.tools.evidence import EvidenceLookup
from aegis.tools.registry import TOOLS

# Require evidence retrieval through the citation-producing lookup.
# The raw fixture loader doesn't produce EvidenceItem IDs.
baseline_tools = [
    tool
    for tool in TOOLS
    if tool.name != "load_fixture_tool"
]

print("Available tools:")
for tool in baseline_tools:
    print("-", tool.name)

Available tools:
- lookup_evidence_tool
- per_90_tool
- normalize_tool
- weighted_score_tool
- rank_tool
- sensitivity_check_tool
- calculator_tool


In [6]:
# The prompt lives in the module so notebooks, tests and experiments share one copy.
from aegis.graphs.baseline import BASELINE_SYSTEM_PROMPT


In [7]:
model = get_model("researcher")

agent = create_agent(
    model=model,
    tools=baseline_tools,
    system_prompt=BASELINE_SYSTEM_PROMPT,
    response_format=ToolStrategy(
        schema=FinalReport,

        # During the first experiment, expose invalid structured output
        # immediately instead of automatically spending on repair attempts.
        handle_errors=False,
    ),
)

print("PASS: baseline agent constructed")

PASS: baseline agent constructed


In [8]:
request = ResearchRequest(
    objective=(
        "Compare Player A and Player B by total goals "
        "in the 2025 Demo League. Do not rank them."
    ),
    subjects=["Player A", "Player B"],
    factors=["scoring"],
    mode="compare",
    scope=ResearchScope(
        time_period="2025",
        coverage="Demo League",
    ),
)

# Reuse the clarification gate before paying for an agent run.
clarification = check_clarity(request)

if clarification.needs_clarification:
    raise ValueError(clarification.model_dump_json(indent=2))

payload = {
    "request": request.model_dump(mode="json"),

    # This selects the test data adapter for this experiment.
    # It does not restrict Aegis's general request schema to football.
    "dataset_ref": "football_demo_v1",
}

print(json.dumps(payload, indent=2))

{
  "request": {
    "objective": "Compare Player A and Player B by total goals in the 2025 Demo League. Do not rank them.",
    "subjects": [
      "Player A",
      "Player B"
    ],
    "factors": [
      "scoring"
    ],
    "mode": "compare",
    "scope": {
      "time_period": "2025",
      "coverage": "Demo League"
    },
    "constraints": [],
    "ranking_basis": null
  },
  "dataset_ref": "football_demo_v1"
}


In [9]:
# Use the smaller of the configured cutoff and $0.05 for this demo.
# model_copy changes only this run's settings object.
run_settings = settings.model_copy(
    update={
        "budget_limit_usd": min(
            settings.budget_limit_usd,
            Decimal("0.05"),
        )
    }
)

tracker, run_config = new_run_budget(run_settings)

run_config.update(
    {
        "run_name": "baseline-football-compare",
        "tags": ["experiment-a", "day-3", "fictional-data"],
        "metadata": {
            "experiment": "A",
            "case_id": "football_compare_2025",
            "dataset_ref": "football_demo_v1",
        },

        # Bound graph execution if the model keeps requesting tools.
        # This is a graph-step limit, not a dollar limit.
        "recursion_limit": 20,
    }
)

try:
    result = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": json.dumps(payload),
                }
            ]
        },
        config=run_config,
    )
finally:
    # Retain visibility into recorded spending even if execution fails.
    print("Recorded model calls:", len(tracker.calls))
    print("Recorded cost USD:", tracker.spent_usd)

Recorded model calls: 3
Recorded cost USD: 0.00142195


In [10]:
tool_names = {tool.name for tool in baseline_tools}
requested_calls = []

for message in result["messages"]:
    if isinstance(message, AIMessage):
        for call in message.tool_calls:
            # Exclude the synthetic FinalReport submission tool.
            if call["name"] in tool_names:
                requested_calls.append(call)

for number, call in enumerate(requested_calls, start=1):
    print(f"{number}. {call['name']}")
    print(json.dumps(call["args"], indent=2))

print("Research/calculation tool calls:", len(requested_calls))

1. lookup_evidence_tool
{
  "subject": "Player A",
  "factor": "scoring",
  "dataset_ref": "football_demo_v1"
}
2. lookup_evidence_tool
{
  "subject": "Player B",
  "factor": "scoring",
  "dataset_ref": "football_demo_v1"
}
Research/calculation tool calls: 2


In [11]:
sources_by_id = {}
evidence_by_id = {}

for message in result["messages"]:
    if not isinstance(message, ToolMessage):
        continue

    if message.name != "lookup_evidence_tool":
        continue

    if getattr(message, "status", "success") == "error":
        raise RuntimeError("An evidence lookup failed")

    # Our current registry returns a dictionary, serialized as JSON
    # in the ToolMessage content.
    if not isinstance(message.content, str):
        raise TypeError("Expected JSON text from lookup_evidence_tool")

    lookup = EvidenceLookup.model_validate_json(message.content)

    if lookup.source is not None:
        sources_by_id[lookup.source.source_id] = lookup.source

    for item in lookup.evidence:
        evidence_by_id[item.evidence_id] = item

bundle = ResearchBundle(
    sources=list(sources_by_id.values()),
    evidence=list(evidence_by_id.values()),
)

report = result["structured_response"]

# Check the generated report still answers the requested task.
if report.mode != request.mode:
    raise ValueError("Report mode differs from the request")

if {s.casefold() for s in report.subjects} != {
    s.casefold() for s in request.subjects
}:
    raise ValueError("Report subjects differ from the request")

validated_report = validate_report_against_bundle(report, bundle)

print(validated_report.model_dump_json(indent=2))
print("PASS: agent report references collected evidence")

{
  "summary": "Comparison of Player A and Player B total goals in 2025 Demo League (compare mode).",
  "mode": "compare",
  "subjects": [
    "Player A",
    "Player B"
  ],
  "comparison_rows": [
    {
      "subject": "Player A",
      "factor": "scoring",
      "value": 14.0,
      "unit": "goals",
      "evidence_ids": [
        "ev_42aa13f70da1973f10be907aa173e5b20984b0b3f94dd6d6671edb4fa10a6519"
      ]
    },
    {
      "subject": "Player B",
      "factor": "scoring",
      "value": 11.0,
      "unit": "goals",
      "evidence_ids": [
        "ev_ca42f0862bb7a926e9972fb79e7e95b69a9621a0d45352a94d1c624fc84ef972"
      ]
    }
  ],
  "ranking": [],
  "findings": [
    {
      "finding_id": "f1",
      "subject": "Player A",
      "factor": "scoring",
      "claim": "Player A scored 14 goals in the 2025 Demo League.",
      "claim_type": "reported_fact",
      "evidence_ids": [
        "ev_42aa13f70da1973f10be907aa173e5b20984b0b3f94dd6d6671edb4fa10a6519"
      ],
      "limitati

In [12]:
from decimal import Decimal

from aegis.graphs.baseline import (
    requested_tool_calls,
    run_baseline_with_details,
)
from aegis.schemas import ResearchRequest, ResearchScope

platform_request = ResearchRequest(
    objective=(
        "Rank Vendor A, Vendor B and Vendor C by storage cost only. "
        "Include each vendor's raw storage price in comparison_rows."
    ),
    subjects=["Vendor A", "Vendor B", "Vendor C"],
    factors=["operating cost"],
    mode="rank",
    scope=ResearchScope(
        time_period="Fictional v1 snapshot",
        coverage="Fictional platform specifications",
    ),
    ranking_basis=(
        "Use storage_usd_tb_month only; lower is better. "
        "Use normalize_tool with direction='lower', then rank_tool. "
        "Use the normalized values as RankingEntry.score. "
        "Map the rank_tool result's 'rank' field to "
        "RankingEntry.position. Ignore compute cost."
    ),
)

In [13]:
platform_run = run_baseline_with_details(
    platform_request,
    dataset_ref="platforms_demo_v1",
    budget_usd=Decimal("0.05"),
    case_id="platform_storage_rank",
)

print(platform_run.report.model_dump_json(indent=2))

print("\nTool calls:")
for index, call in enumerate(
    requested_tool_calls(platform_run.messages),
    start=1,
):
    print(index, call["name"], call["args"])

print("\nModel calls:", len(platform_run.tracker.calls))
print("Cost USD:", platform_run.tracker.spent_usd)

{
  "summary": "Ranked Vendor A, Vendor B, Vendor C by storage cost (storage_usd_tb_month) from the Fictional v1 snapshot. Lower storage cost is better; normalized with direction='lower' and ranked by descending normalized score. Report includes raw prices and evidence IDs. Findings are unverified and data is synthetic (dataset flagged as test data).",
  "mode": "rank",
  "subjects": [
    "Vendor A",
    "Vendor B",
    "Vendor C"
  ],
  "comparison_rows": [
    {
      "subject": "Vendor A",
      "factor": "operating cost",
      "value": 20.0,
      "unit": "USD per TB per month",
      "evidence_ids": [
        "ev_036f9d325876f88a499db81dd8431128c26eb335feaa5b29ab41de674831a036"
      ]
    },
    {
      "subject": "Vendor B",
      "factor": "operating cost",
      "value": 25.0,
      "unit": "USD per TB per month",
      "evidence_ids": [
        "ev_941a2b38cc4a74feaf8b3b557adbc15a506815915c75c4266e8026c6e48551c6"
      ]
    },
    {
      "subject": "Vendor C",
      "fact

In [14]:
import math

# Independently calculated expectations—not generated by the agent.
expected_ranking = {
    "Vendor C": (1, 1.0),
    "Vendor A": (2, 0.5),
    "Vendor B": (3, 0.0),
}

actual_ranking = {
    entry.subject: entry
    for entry in platform_run.report.ranking
}

assert platform_run.report.status == "completed"
assert set(actual_ranking) == set(expected_ranking)

for subject, (position, score) in expected_ranking.items():
    entry = actual_ranking[subject]

    assert entry.position == position
    assert math.isclose(
        entry.score,
        score,
        rel_tol=0,
        abs_tol=1e-9,
    )

# Verify that the requested calculation tools actually completed.
completed_tools = {
    message.name
    for message in platform_run.messages
    if isinstance(message, ToolMessage)
    and getattr(message, "status", "success") == "success"
}

assert {
    "lookup_evidence_tool",
    "normalize_tool",
    "rank_tool",
} <= completed_tools

print("PASS: platform ranking matches the independent calculation")

PASS: platform ranking matches the independent calculation
